## K-Nearest Neighbors

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

breast_data = pd.read_excel('./Data/breast-cancer-wisconsin.xlsx')
breast_data.head()


In [ ]:
breast_data.Class.unique()

In [ ]:
breast_data.describe()

In [ ]:
breast_data['Class'].value_counts()

Lo primero que debemos hacer, es dividir nuestros datos para crear el set de train y de test, para esto usamos la función `train_test_split` que viene en `sklearn.model_selection`. Debemos recordar que al K-NN estar basado en distancias, por lo que sería necesario estandarizar o normalizar los datos antes de ponerlos en el modelo.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

breast_data.drop(['ID'], axis='columns',inplace=True)
breast_X =  breast_data.drop(['Class'], axis='columns')
breast_y = breast_data['Class']

scaler = MinMaxScaler().fit(breast_X)
scaled_breast_X = pd.DataFrame(scaler.transform(breast_X))
scaled_breast_X.head()

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(breast_X, breast_y, test_size=0.2, stratify = breast_y, random_state=2023)

In [ ]:
display(train_y.value_counts())
display(test_y.value_counts())

Ahora ajustamos el modelo de K-NN

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors = 3)
knn_model.fit(train_X,train_y)

Veamos que tal lo hace el modelo, para eso realizaremos predicciones y luego imprimiremos la matriz de confusión

In [ ]:
knn_model.predict_proba(test_X)

In [ ]:
knn_model.predict(test_X)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_values = knn_model.predict(test_X)
print(classification_report(test_y,pred_values))
print(confusion_matrix(test_y,pred_values))

Veamos con otro set de datos

In [ ]:
heart = pd.read_excel('./Data/Heart.xlsx')
heart.head()

In [ ]:
heart.dropna(inplace=True)
heart_X = heart.drop(['ChestPain','Thal','AHD'],axis='columns')
heart_y = heart['AHD'].replace(('Yes','No'),(1,0))

In [ ]:
minmax_ahd = MinMaxScaler().fit(heart_X)
scaled_heart_df = pd.DataFrame(minmax_ahd.transform(heart_X), columns=heart_X.columns)
scaled_heart_df.head()

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(heart_X, heart_y, test_size=0.2, stratify=heart_y,random_state=2023)

In [ ]:
heart_y.value_counts()

In [ ]:
160/(160+137)

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors = 2)
knn_model.fit(train_X,train_y)

pred_values = knn_model.predict(test_X)
print(classification_report(test_y,pred_values))
print(confusion_matrix(test_y,pred_values))

veamos qué pasa cuando optimizamos los hiper-parámetros del modelo utilizando `GridSearch`

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors' : [2, 3, 5, 7, 9, 11, 15],
    'weights' : ['uniform','distance'],
    'metric' : ['euclidean','manhattan']
}

opt_knn_model = GridSearchCV(KNeighborsClassifier(), 
                             param_grid = param_grid, 
                             n_jobs=-1,
                             cv = 5,
                             scoring = 'accuracy')

opt_knn_model.fit(train_X,train_y)
pred_values_knn = opt_knn_model.predict(test_X)

print(opt_knn_model.best_params_)
print(opt_knn_model.best_score_)
print(classification_report(test_y,pred_values_knn))
print(confusion_matrix(test_y,pred_values_knn))


In [ ]:
param_grid = {
    'n_neighbors' : range(2,25),
    'weights' : ['uniform','distance'],
    'metric' : ['euclidean','manhattan']
}

opt_knn_model = GridSearchCV(KNeighborsClassifier(), 
                             param_grid = param_grid, 
                             n_jobs=-1,
                             cv = 5,
                             scoring = 'accuracy')

opt_knn_model.fit(train_X,train_y)
pred_values_knn = opt_knn_model.predict(test_X)

print(opt_knn_model.best_params_)
print(opt_knn_model.best_score_)
print(classification_report(test_y,pred_values_knn))
print(confusion_matrix(test_y,pred_values_knn))

In [ ]:
opt_knn_model.cv_results_

# Naive Bayes

Veamos ahora como se desempeña en este último set de datos el modelo de Naive Bayes

In [ ]:
from sklearn.naive_bayes import BernoulliNB, MultinomialNB

naive_model = BernoulliNB()
naive_model.fit(train_X,train_y)
pred_values_nb = naive_model.predict(test_X)

print(classification_report(test_y,pred_values_nb))
print(confusion_matrix(test_y,pred_values_nb))

In [ ]:
#hasta el momento
from sklearn.metrics import f1_score

print(f"K-NN: {f1_score(test_y, pred_values_knn)}")
print(f"Naive Bayes: {f1_score(test_y, pred_values_nb)}")

# SVM

Es el turno de support vector machine

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel = 'linear', C = 100) # C parameter, kernel
svm_model.fit(train_X,train_y)
pred_values_svm = svm_model.predict(test_X)

print(classification_report(test_y,pred_values_svm))
print(confusion_matrix(test_y,pred_values_svm))

arreglemos un poquito más el modelo de SVM

In [ ]:
param_grid = {
    #'degree' : [2,3,4,5,6,7,8,10,15,20,30,50],
    'kernel' : ['linear','rbf'],
    'C' : [0.01,0.1,1,10,100]
}

opt_svm_model = GridSearchCV(SVC(), param_grid = param_grid, cv = 5, n_jobs=-1)
opt_svm_model.fit(train_X,train_y)
pred_values_osvm = opt_svm_model.predict(test_X)

print(opt_svm_model.best_params_)
print(classification_report(test_y,pred_values_osvm))
print(confusion_matrix(test_y,pred_values_osvm))

In [ ]:
#hasta el momento
from sklearn.metrics import f1_score

print(f"K-NN: {f1_score(test_y, pred_values_knn)}")
print(f"Naive Bayes: {f1_score(test_y, pred_values_nb)}")
print(f"SVM: {f1_score(test_y, pred_values_osvm)}")

# Árboles de decisión

In [ ]:
train_y

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree_model = DecisionTreeClassifier(max_depth=4)
tree_model.fit(train_X, train_y)
pred_values_tree = tree_model.predict(test_X)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(test_y,pred_values_tree))
print(confusion_matrix(test_y,pred_values_tree))

revisemos como es el árbol

In [ ]:
from matplotlib import pyplot as plt

plt.figure(figsize=(20,20))
features = heart.columns
classes = ['Not heart disease','heart disease']
plot_tree(tree_model,feature_names=features,class_names=classes,filled=True)
plt.show()

In [ ]:
tree_model.feature_importances_

In [ ]:
tree_model.feature_names_in_

podemos hacer este árbol un poco mejor

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {'max_depth': [2,4,6,8,10,12],
         'min_samples_split': [5,10,15,20],
         'min_samples_leaf': [5,10,15,20]}

opt_tree_model = GridSearchCV(DecisionTreeClassifier(),param_grid=params, cv = 5, n_jobs=-1)
opt_tree_model.fit(train_X,train_y)
pred_values_otree = opt_tree_model.predict(test_X)

print(opt_tree_model.best_params_)
print(classification_report(test_y,pred_values_otree))
print(confusion_matrix(test_y,pred_values_otree))

veamos como queda el árbol ahora

In [ ]:
best_tree_model = opt_tree_model.best_estimator_
best_tree_model.fit(train_X,train_y)

plt.figure(figsize=(20,20))
features = heart.columns
classes = ['Not heart disease','heart disease']
plot_tree(best_tree_model,feature_names=features,class_names=classes,filled=True)
plt.show()

In [ ]:
#hasta el momento
from sklearn.metrics import f1_score

print(f"K-NN: {f1_score(test_y, pred_values_knn)}")
print(f"Naive Bayes: {f1_score(test_y, pred_values_nb)}")
print(f"SVM: {f1_score(test_y, pred_values_osvm)}")
print(f"Tree: {f1_score(test_y, pred_values_otree)}")

## Bagging

In [ ]:
from sklearn.ensemble import BaggingClassifier

bag_model = BaggingClassifier()
bag_model.fit(train_X, train_y)

pred_values_bag = bag_model.predict(test_X)
print(classification_report(test_y,pred_values_bag))
print(confusion_matrix(test_y,pred_values_bag))

In [ ]:
from sklearn.metrics import f1_score

print(f"K-NN: {f1_score(test_y, pred_values_knn)}")
print(f"Naive Bayes: {f1_score(test_y, pred_values_nb)}")
print(f"SVM: {f1_score(test_y, pred_values_osvm)}")
print(f"Tree: {f1_score(test_y, pred_values_otree)}")
print(f"Bagging: {f1_score(test_y, pred_values_bag)}")

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=1500, oob_score=True)
rf_model.fit(train_X, train_y)

pred_values_rf = rf_model.predict(test_X)
print(classification_report(test_y,pred_values_rf))
print(confusion_matrix(test_y,pred_values_rf))

In [ ]:
rf_model.oob_score_

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators' : [100,500,1000],
    'max_depth' : [1,2,3,5,7],
    'max_features' : [2,3,5,7,9]
}

opt_rf_model = GridSearchCV(RandomForestClassifier(), param_grid = param_grid, n_jobs = -1, cv = 5)
opt_rf_model.fit(train_X,train_y)
pred_values_rf = opt_rf_model.predict(test_X)

print(opt_rf_model.best_params_)
print(classification_report(test_y,pred_values_rf))
print(confusion_matrix(test_y,pred_values_rf))

In [ ]:
from sklearn.metrics import f1_score

print(f"K-NN: {f1_score(test_y, pred_values_knn)}")
print(f"Naive Bayes: {f1_score(test_y, pred_values_nb)}")
print(f"SVM: {f1_score(test_y, pred_values_osvm)}")
print(f"Tree: {f1_score(test_y, pred_values_otree)}")
print(f"Bagging: {f1_score(test_y, pred_values_bag)}")
print(f"Random Forest: {f1_score(test_y, pred_values_rf)}")

## Regresión

In [ ]:
from sklearn.ensemble import RandomForestRegressor

housing = pd.read_csv('./Data/ames_housing.csv')
housing.drop(['Unnamed: 0'], axis = 'columns', inplace=True)
housing.head()

In [ ]:
numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
housing = housing.select_dtypes(include=numerics)
print(housing.shape)
print(housing.head())

In [ ]:
housing.drop(['Longitude','Latitude'], axis = 'columns', inplace=True)
housing.head()

In [ ]:
housing_X = housing.drop(['Sale_Price'],axis = 'columns')
housing_y = housing.Sale_Price

train_reg_X, test_reg_X, train_reg_y, test_reg_y = train_test_split(housing_X,housing_y, test_size=0.2)

In [ ]:
rf_regressor = RandomForestRegressor(n_estimators=500)
rf_regressor.fit(train_reg_X,train_reg_y)
pred_values_rf_reg = rf_regressor.predict(test_reg_X)

In [ ]:
rf_regressor.score(test_reg_X,test_reg_y)